# 05 - Run Classical Baseline Optimizers (CMA-ES, DE, PSO)

This notebook benchmarks the classical baseline algorithms (**CMA-ES**, **Differential Evolution (DE)**, and **Particle Swarm Optimization (PSO)**) across all experimental conditions ($D \in \{2, 3, 5\}$, $\sigma \in \{0.0, 0.05\}$, $f_i \in \{1, 8, 11, 15, 21\}$) over $N=10$ independent instances.

### Key Principles & Synergy with Notebook 04:
1. **Unified Storage**: Outputs directly to `results/evaluations/{dim}D/std_{noise_std}/f{problem_id}/{algo_name}/`.
2. **Pre-Flight Diagnostics**: Inspects existing data on disk and skips completed/valid evaluations in milliseconds.
3. **Standardized Telemetry**: Attaches `ioh.logger.Analyzer` across instances $1 \dots 10$ and generates `provenance.json`.
4. **Ground-Truth Precision**: Evaluates candidate solutions against clean objectives to prevent noisy ranking bias.


In [ ]:
import sys
import json
import time
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import cma
from scipy.optimize import differential_evolution
from IPython.display import display, HTML
import ioh

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from core.config import DATA_DIR, PROJECT_ROOT, RESULTS_DIR
from domain.services.noise_strategy import MultiplicativeNoiseStrategy, NoNoiseStrategy
from infra.problems.bbob import BBOBProblem
from infra.storage import get_db_connection

# ── User Execution Controls ───────────────────────────────────────────────────
FORCE_REEVALUATE = False            # Set to True to re-run and overwrite existing completed logs
FILTER_BASELINES = None             # e.g., ['pso'] or ['cmaes', 'de'] (None evaluates all)
FILTER_PROBLEMS  = None             # e.g., [1, 8] (None evaluates all)
FILTER_DIMS      = None             # e.g., [2, 3] (None evaluates all)
FILTER_NOISE     = None             # e.g., [0.0, 0.05] (None evaluates all)
N_RUNS           = 10               # Independent runs per condition (instances 1..10)

EVALUATIONS_DIR = RESULTS_DIR / 'evaluations'

# ── Discover Target Conditions from Database ─────────────────────────────────
with get_db_connection() as conn:
    df_db = pd.read_sql_query(
        "SELECT DISTINCT dim, noise_std, problem_id, budget FROM experiments WHERE status = 'completed' ORDER BY dim, noise_std, problem_id",
        conn
    )

if df_db.empty:
    raise RuntimeError("No completed experiments found in database. Please run Notebook 02 first.")

BUDGET = int(df_db['budget'].dropna().max())
UNIQUE_CONFIGS = [
    (int(r['dim']), float(r['noise_std']), int(r['problem_id']))
    for _, r in df_db.iterrows()
]

print(f"🎯 Discovered {len(UNIQUE_CONFIGS)} target experimental conditions across D={sorted(df_db['dim'].unique())}, f={sorted(df_db['problem_id'].unique())}.")


## 1. Classical Baseline Algorithms (CMA-ES, DE, PSO)
Implements standard optimization algorithms conforming to the budget and bounds contract.

In [ ]:
def run_cmaes(problem: BBOBProblem, dim: int, budget: int):
    """Run Covariance Matrix Adaptation Evolution Strategy (CMA-ES)."""
    x0 = [0.0] * dim
    sigma0 = 2.0
    opts = {'bounds': [-5.0, 5.0], 'verbose': -9, 'maxfevals': budget}
    es = cma.CMAEvolutionStrategy(x0, sigma0, opts)
    while not es.stop():
        solutions = es.ask()
        es.tell(solutions, [problem(x) for x in solutions])
        if problem.evaluations >= budget:
            break
    return es.result.xbest, es.result.fbest

def run_de(problem: BBOBProblem, dim: int, budget: int):
    """Run Differential Evolution (DE)."""
    bounds = [(-5.0, 5.0)] * dim
    maxiter = max(1, budget // (15 * dim))
    
    def obj_fn(x):
        if problem.evaluations >= budget:
            raise StopIteration('Budget exhausted')
        return problem(x)
        
    try:
        res = differential_evolution(obj_fn, bounds, maxiter=maxiter, seed=None)
        return res.x, res.fun
    except StopIteration:
        return problem.optimum_x, problem.true_optimum

def run_pso(problem: BBOBProblem, dim: int, budget: int, n_particles: int = 30, w: float = 0.729, c1: float = 1.49445, c2: float = 1.49445):
    """Run Particle Swarm Optimization (PSO)."""
    lb, ub = problem.lower_bound, problem.upper_bound
    X = np.random.uniform(lb, ub, (n_particles, dim))
    V = np.random.uniform(-abs(ub - lb), abs(ub - lb), (n_particles, dim)) * 0.1
    
    pbest_X = X.copy()
    pbest_y = np.array([problem(x) for x in X])
    
    gbest_idx = np.argmin(pbest_y)
    gbest_X = pbest_X[gbest_idx].copy()
    gbest_y = pbest_y[gbest_idx]
    
    evals = n_particles
    while evals < budget:
        r1 = np.random.rand(n_particles, dim)
        r2 = np.random.rand(n_particles, dim)
        V = w * V + c1 * r1 * (pbest_X - X) + c2 * r2 * (gbest_X - X)
        X = np.clip(X + V, lb, ub)
        
        for i in range(n_particles):
            if evals >= budget:
                break
            y = problem(X[i])
            evals += 1
            if y < pbest_y[i]:
                pbest_y[i] = y
                pbest_X[i] = X[i].copy()
                if y < gbest_y:
                    gbest_y = y
                    gbest_X = X[i].copy()
                    
    return gbest_X, gbest_y

BASELINES = {
    'cmaes': ('CMA-ES', run_cmaes),
    'de':    ('Differential Evolution', run_de),
    'pso':   ('Particle Swarm Optimization', run_pso),
}

print(f"✅ Configured {len(BASELINES)} baseline algorithms: {', '.join(BASELINES.keys())}")


## 2. Pre-Flight Diagnostic Dashboard: Baseline Coverage & Workload Audit
Scans `results/evaluations/` across all 30 conditions to categorize what is already completed vs queued.

In [ ]:
audit_records = []

for dim, noise_std, p_id in UNIQUE_CONFIGS:
    for algo_slug, (algo_display, _) in BASELINES.items():
        is_filtered = False
        if FILTER_BASELINES and algo_slug not in [b.lower() for b in FILTER_BASELINES]: is_filtered = True
        if FILTER_PROBLEMS and p_id not in FILTER_PROBLEMS: is_filtered = True
        if FILTER_DIMS and dim not in FILTER_DIMS: is_filtered = True
        if FILTER_NOISE and noise_std not in FILTER_NOISE: is_filtered = True

        target_folder = EVALUATIONS_DIR / f"{dim}D" / f"std_{noise_std}" / f"f{p_id}" / algo_slug
        prov_path = target_folder / "provenance.json"
        
        status = 'PENDING'
        runs_found = 0
        med_err = None

        if target_folder.exists():
            dat_files = [f for f in target_folder.glob('**/*.dat') if f.stat().st_size > 0]
            runs_found = len(dat_files)
            if prov_path.exists() and runs_found > 0:
                try:
                    prov = json.loads(prov_path.read_text(encoding='utf-8'))
                    med_err = prov.get('median_clean_error')
                    status = 'COMPLETED'
                except Exception:
                    status = 'NEEDS_RERUN'
            elif runs_found > 0:
                status = 'COMPLETED'
            else:
                status = 'PENDING'

        audit_records.append({
            'baseline': algo_slug,
            'display_name': algo_display,
            'dim': dim,
            'noise_std': noise_std,
            'problem_id': p_id,
            'status': status,
            'runs_found': runs_found,
            'median_error': med_err,
            'is_filtered': is_filtered,
        })

df_audit_base = pd.DataFrame(audit_records)

# Summary by baseline
summary_rows = []
for b_name, grp in df_audit_base.groupby('baseline'):
    tot = len(grp)
    comp = len(grp[grp['status'] == 'COMPLETED'])
    pend = len(grp[grp['status'] == 'PENDING'])
    rerun = len(grp[grp['status'] == 'NEEDS_RERUN'])
    to_run = len(grp[(grp['status'].isin(['PENDING', 'NEEDS_RERUN'])) & (~grp['is_filtered'])])
    pct = (comp / tot * 100) if tot > 0 else 0.0
    summary_rows.append({
        'Baseline': b_name.upper(),
        'Total Tasks': tot,
        'Completed': comp,
        'Pending': pend,
        'Needs Rerun': rerun,
        'Queue to Run': to_run,
        'Progress (%)': pct,
    })

df_summary_base = pd.DataFrame(summary_rows)
tot_queue = int(df_summary_base['Queue to Run'].sum())
tot_comp = int(df_summary_base['Completed'].sum())
tot_tasks = int(df_summary_base['Total Tasks'].sum())
ov_pct = (tot_comp / tot_tasks * 100) if tot_tasks > 0 else 0.0

# Render Clean HTML Visual Dashboard
html_dash = f"""
<div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 950px; margin: 15px 0;">
    <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 16px; border-bottom: 2px solid #E2E8F0; padding-bottom: 8px;">
        <div>
            <h2 style="margin: 0; color: #0F172A; font-size: 20px; font-weight: 700; display: flex; align-items: center; gap: 8px;">
                📊 Classical Baselines Pre-Flight Audit
            </h2>
            <p style="margin: 4px 0 0 0; color: #64748B; font-size: 13px;">Status of empirical benchmark runs for CMA-ES, DE, and PSO across 30 conditions</p>
        </div>
        <div style="background: #EEF2F6; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; color: #334155;">
            Target: N={N_RUNS} runs / condition
        </div>
    </div>

    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 20px;">
        <div style="background: linear-gradient(135deg, #1E293B 0%, #0F172A 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #94A3B8;">Total Baseline Tasks</div>
            <div style="font-size: 24px; font-weight: 700; color: #F8FAFC; margin-top: 4px;">{tot_tasks}</div>
            <div style="font-size: 11px; color: #64748B; margin-top: 2px;">3 Optimizers × {len(UNIQUE_CONFIGS)} Conditions</div>
        </div>
        <div style="background: linear-gradient(135deg, #065F46 0%, #047857 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #A7F3D0;">Completed & Valid</div>
            <div style="font-size: 24px; font-weight: 700; color: #ECFDF5; margin-top: 4px;">{tot_comp}</div>
            <div style="font-size: 11px; color: #D1FAE5; margin-top: 2px;">{ov_pct:.1f}% Baseline Progress</div>
        </div>
        <div style="background: linear-gradient(135deg, #C2410C 0%, #9A3412 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #FED7AA;">Queue to Run</div>
            <div style="font-size: 24px; font-weight: 700; color: #FFF7ED; margin-top: 4px;">{tot_queue}</div>
            <div style="font-size: 11px; color: #FFEDD5; margin-top: 2px;">{tot_queue * N_RUNS} Total Runs</div>
        </div>
        <div style="background: linear-gradient(135deg, #4338CA 0%, #3730A3 100%); padding: 14px 18px; border-radius: 10px; color: white; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 0.05em; color: #C7D2FE;">Est. Runtime</div>
            <div style="font-size: 24px; font-weight: 700; color: #EEF2FF; margin-top: 4px;">~{tot_queue * 0.8:.0f}m</div>
            <div style="font-size: 11px; color: #E0E7FF; margin-top: 2px;">@ ~0.8s per condition</div>
        </div>
    </div>

    <div style="background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 10px; padding: 16px; margin-bottom: 8px;">
        <div style="font-size: 13px; font-weight: 700; color: #1E293B; margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.04em;">
            Baseline Completion Progress
        </div>
"""

for _, row in df_summary_base.iterrows():
    b_name = row['Baseline']
    tot = int(row['Total Tasks'])
    comp = int(row['Completed'])
    pend = int(row['Pending'])
    rerun = int(row['Needs Rerun'])
    q = int(row['Queue to Run'])
    pct = float(row['Progress (%)'])
    color = '#10B981' if pct > 75 else '#3B82F6' if pct > 25 else '#F59E0B'
    rerun_pct = (rerun / tot * 100) if tot > 0 else 0
    
    html_dash += f"""
        <div style="margin-bottom: 14px;">
            <div style="display: flex; justify-content: space-between; font-size: 13px; font-weight: 600; color: #334155; margin-bottom: 4px;">
                <span>⚙️ <strong style="color: #0F172A;">{b_name}</strong> &nbsp;({comp}/{tot} Completed)</span>
                <span style="color: {color}; font-weight: 700;">{pct:.1f}%</span>
            </div>
            <div style="background: #E2E8F0; border-radius: 6px; height: 10px; overflow: hidden; display: flex;">
                <div style="background: #10B981; width: {pct}%; transition: width 0.3s;"></div>
                <div style="background: #EF4444; width: {rerun_pct}%;"></div>
            </div>
            <div style="display: flex; gap: 14px; font-size: 11px; color: #64748B; margin-top: 5px;">
                <span>✅ Completed: <strong style="color: #059669;">{comp}</strong></span>
                <span>⏳ Pending: <strong style="color: #D97706;">{pend}</strong></span>
                <span>⚠️ Needs Rerun: <strong style="color: #DC2626;">{rerun}</strong></span>
                <span>🎯 Queue: <strong style="color: #2563EB;">{q}</strong></span>
            </div>
        </div>
    """

html_dash += """
    </div>
</div>
"""

display(HTML(html_dash))


## 3. Execute Baseline Benchmarks (N=10 Independent Runs)
Runs all queued classical optimizers with `ioh.logger.Analyzer` and writes `provenance.json`.

In [ ]:
def write_baseline_provenance(prov_path: Path, algo_name: str, dim: int, noise_std: float, p_id: int, med_err: float):
    prov = {
        'algorithm_name':     algo_name,
        'solver_type':        'classical_baseline',
        'dim':                dim,
        'noise_std':          noise_std,
        'problem_id':         p_id,
        'n_runs':             N_RUNS,
        'budget':             BUDGET,
        'median_clean_error': float(med_err) if not np.isinf(med_err) else None,
        'evaluated_at':       pd.Timestamp.now().isoformat(),
    }
    prov_path.write_text(json.dumps(prov, indent=2), encoding='utf-8')

# Build evaluation queue
queue = [
    (dim, noise_std, p_id, algo_slug, algo_name, algo_fn)
    for dim, noise_std, p_id in UNIQUE_CONFIGS
    for algo_slug, (algo_name, algo_fn) in BASELINES.items()
    if not (
        (FILTER_BASELINES and algo_slug not in [b.lower() for b in FILTER_BASELINES])
        or (FILTER_PROBLEMS and p_id not in FILTER_PROBLEMS)
        or (FILTER_DIMS and dim not in FILTER_DIMS)
        or (FILTER_NOISE and noise_std not in FILTER_NOISE)
    )
]

print(f'=== Starting Baseline Benchmark Engine ({len(queue)} candidate configurations in scope) ===\n')
start_time = time.time()
evaluated_count = 0
skipped_count = 0
failed_count = 0

for idx, (dim, noise_std, p_id, algo_slug, algo_name, algo_fn) in enumerate(queue, start=1):
    out_dir = EVALUATIONS_DIR / f"{dim}D" / f"std_{noise_std}" / f"f{p_id}"
    target_log_folder = out_dir / algo_slug
    prov_path = target_log_folder / "provenance.json"

    # Cache Validation: Check if valid non-empty .dat files exist
    if not FORCE_REEVALUATE and target_log_folder.exists():
        dat_files = [f for f in target_log_folder.glob('**/*.dat') if f.stat().st_size > 0]
        if len(dat_files) > 0:
            skipped_count += 1
            continue

    print(f'  [{idx:2d}/{len(queue):2d}] ⚡ Running [{algo_slug.upper()}] on f{p_id} ({dim}D, noise={noise_std}) across {N_RUNS} runs...')
    
    out_dir.mkdir(parents=True, exist_ok=True)
    if target_log_folder.exists():
        shutil.rmtree(target_log_folder, ignore_errors=True)

    noise_strat = MultiplicativeNoiseStrategy(noise_std) if noise_std > 0.0 else NoNoiseStrategy()
    run_errors = []
    t0 = time.time()

    logger = ioh.logger.Analyzer(
        root=str(out_dir),
        folder_name=algo_slug,
        algorithm_name=algo_name,
        store_positions=False
    )

    for run_idx in range(1, N_RUNS + 1):
        problem = BBOBProblem(
            problem_id=p_id,
            dim=dim,
            instance_id=run_idx,
            noise_strategy=noise_strat,
        )
        problem.attach_logger(logger)

        try:
            best_x, best_y = algo_fn(problem, dim, BUDGET)
            clean_prob = BBOBProblem(problem_id=p_id, dim=dim, instance_id=run_idx, noise_strategy=NoNoiseStrategy())
            final_err = clean_prob(best_x) if best_x is not None else float(best_y)
            run_errors.append(final_err)
        except Exception as e:
            print(f'       ❌ Run {run_idx:2d}/{N_RUNS} failed: {e}')
            run_errors.append(float('inf'))
        finally:
            if hasattr(problem, 'clean_problem') and hasattr(problem.clean_problem, 'detach_logger'):
                problem.clean_problem.detach_logger()

    del logger
    med_err = np.median(run_errors) if run_errors else float('inf')
    write_baseline_provenance(prov_path, algo_name, dim, noise_std, p_id, med_err)
    elapsed = time.time() - t0
    evaluated_count += 1
    print(f'       ✅ Done in {elapsed:.1f}s | Median Error: {med_err:.4e}')

total_elapsed = time.time() - start_time
print('\n' + '='*80)
print(f'🎯 Baseline Evaluation Complete in {total_elapsed:.1f}s ({evaluated_count} evaluated, {skipped_count} skipped/cached, {failed_count} errors)')
print('👉 Run Notebook 06 (06_experimental_audit.ipynb) for the full matrix audit.')
print('='*80)
